# Solutions · 02 · Monte Carlo estimation and its error

Worked solutions to the exercises of [notebook 02](../notebooks/02_monte_carlo_estimation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from engstoch import montecarlo as mc, output as oa
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Estimate pi by throwing points in the unit square, with a 99 % interval, and find the number of points needed for six correct digits. How long would that take at 10^8 points per second?

In [2]:
n = 10**6
u = R(1).random((n, 2))
hit = (np.sum(u**2, axis=1) <= 1).astype(float) * 4
r = mc.mean_ci(hit, level=0.99)
print(f"pi ~ {r['estimate']:.5f}, 99 % CI ({r['ci'][0]:.5f}, {r['ci'][1]:.5f}); sigma = {r['std_dev']:.4f}")
n6 = mc.sample_size(r["std_dev"], 0.5e-5, 0.99)
print(f"six correct digits (half-width 5e-6 at 99 %): n = {n6:.2e} points, {n6 / 1e8 / 3600:.1f} hours at 1e8 points per second")

pi ~ 3.14320, 99 % CI (3.13897, 3.14742); sigma = 1.6411
six correct digits (half-width 5e-6 at 99 %): n = 7.15e+11 points, 2.0 hours at 1e8 points per second


Each point gives 4·1{inside}, with standard deviation 4√(p(1 − p)) ≈ 1.64. Six digits need a half-width of
5·10⁻⁶, i.e. n ≈ (2.58 × 1.64 / 5·10⁻⁶)² ≈ 7·10¹¹ points - about two hours at a hundred million per second. Monte
Carlo is the wrong tool for computing π; it is the right one when there is no better method.

## Exercise 2

For the lognormal(0, 1.5) mean, compare the coverage at n = 1000 of the t interval and of a percentile bootstrap interval (2000 resamples) over 500 replications. Does the bootstrap fix the problem?

In [3]:
truth = np.exp(1.125)
rows = []
cov_t, cov_b = [], []
for k in range(500):
    x = R(100 + k).lognormal(0, 1.5, 1000)
    cov_t.append(mc.mean_ci(x, method="t")["ci"])
    boot = R(9000 + k).choice(x, (2000, len(x))).mean(axis=1)
    cov_b.append(tuple(np.quantile(boot, [0.025, 0.975])))
print("t interval:          coverage", oa.coverage([a for a, b in cov_t], [b for a, b in cov_t], truth))
print("percentile bootstrap: coverage", oa.coverage([a for a, b in cov_b], [b for a, b in cov_b], truth))

t interval:          coverage {'coverage': 0.926, 'std_error': 0.011706750189527404}
percentile bootstrap: coverage {'coverage': 0.924, 'std_error': 0.01185107590052481}


The bootstrap does not fix it: both intervals cover the true mean about 92-93 % of the time instead of 95 %.
The bootstrap resamples the observed data, so it cannot know about the large values that a sample of 1000
rarely contains - the source of the problem. For heavy tails one needs more data, a parametric model for the
tail, or a transformation (estimate the lognormal parameters and transform back).

## Exercise 3

**Implement it yourself:** write the Wilson score interval for a proportion and use it to show by simulation that, for p = 0.001 and n = 2000, the Wald interval's coverage is far below 95 % while Wilson's is close.

In [4]:
p, n = 0.001, 2000
z = 1.959963984540054
cw, cwi = [], []
for k in range(20_000):
    s = R(k).binomial(n, p)
    ph = s / n
    half = z * np.sqrt(ph * (1 - ph) / n)
    cw.append(ph - half <= p <= ph + half)
    centre = (ph + z**2 / (2 * n)) / (1 + z**2 / n)
    hw = z * np.sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    cwi.append(centre - hw <= p <= centre + hw)
print(f"coverage at p = 0.001, n = 2000: Wald {np.mean(cw):.3f}, Wilson {np.mean(cwi):.3f}")
print("library Wilson for 2 successes:", mc.proportion_ci(2, 2000)["ci"])

coverage at p = 0.001, n = 2000: Wald 0.862, Wilson 0.948
library Wilson for 2 successes: (0.00027427891765203377, 0.003638934269038393)


With np = 2 the count is 0 in 13.5 % of runs, and the Wald interval is then the single point 0 - it can never
cover. Its coverage is about 86 %; Wilson's interval, which inverts the score test rather than plugging in p̂,
stays near 95 %. For rare-event estimation by crude Monte Carlo, Wald intervals are unusable.